# Value Iteration on a Stochastic Grid World

A 4x4 grid world solved with value iteration. The agent picks a direction, but the move only goes the way it meant with probability `p`. The rest of the probability is split evenly between the two perpendicular directions. The notebook solves the grid for `p = 0.7`, then sweeps `p` from 0.1 to 0.9 to see how the optimal policy changes.

Grid layout, with `(x, y)` and `(0, 0)` in the bottom-left corner:

```
y=3   WALL    .     -1    +1
y=2    .      .      .     .
y=1    .     WALL    .    WALL
y=0    .      .      .     .
      x=0    x=1    x=2   x=3
```

- Every move costs `-0.04`, and the discount factor is `0.95`.
- Moving into a wall or off the grid leaves the agent where it is.
- `+1` and `-1` are terminal states.

## Setup

In [1]:
from dataclasses import dataclass

GAMMA = 0.95
STEP_COST = -0.04
THRESHOLD = 1e-4

WIDTH = 4
HEIGHT = 4

WALLS = {(0, 3), (1, 1), (3, 1)}
TERMINAL_REWARDS = {
    (2, 3): -1.0,  # penalty
    (3, 3): 1.0,   # reward
}

ACTIONS = ("up", "down", "left", "right")

DELTA = {
    "up": (0, 1),
    "down": (0, -1),
    "left": (-1, 0),
    "right": (1, 0),
}

PERPENDICULAR = {
    "up": ("left", "right"),
    "down": ("left", "right"),
    "left": ("up", "down"),
    "right": ("up", "down"),
}


## Grid world and the transition model

In [2]:
@dataclass(frozen=True)
class GridWorld:
    width: int = WIDTH
    height: int = HEIGHT

    def in_bounds(self, state):
        x, y = state
        return 0 <= x < self.width and 0 <= y < self.height

    def is_wall(self, state):
        return state in WALLS

    def is_terminal(self, state):
        return state in TERMINAL_REWARDS

    def is_non_sink_state(self, state):
        return not self.is_wall(state) and not self.is_terminal(state)

    def all_states(self):
        return [(x, y) for y in range(self.height) for x in range(self.width)]

    def step(self, state, action):
        dx, dy = DELTA[action]
        nxt = (state[0] + dx, state[1] + dy)
        if not self.in_bounds(nxt) or self.is_wall(nxt):
            return state
        return nxt


def transition_distribution(action, p):
    side_prob = (1.0 - p) / 2.0
    left_right = PERPENDICULAR[action]
    return {
        action: p,
        left_right[0]: side_prob,
        left_right[1]: side_prob,
    }


def expected_utility(grid, state, action, utilities, p):
    trans = transition_distribution(action, p)
    future = 0.0
    for move_dir, prob in trans.items():
        nxt = grid.step(state, move_dir)
        future += prob * utilities[nxt]
    return STEP_COST + GAMMA * future


def print_utility_grid(utilities):
    for y in range(HEIGHT - 1, -1, -1):
        row = []
        for x in range(WIDTH):
            s = (x, y)
            if s in WALLS:
                row.append("  WALL   ")
            elif s in TERMINAL_REWARDS:
                row.append(f"{TERMINAL_REWARDS[s]:8.4f}")
            else:
                row.append(f"{utilities[s]:8.4f}")
        print(" ".join(row))


def print_policy_grid(policy):
    for y in range(HEIGHT - 1, -1, -1):
        row = []
        for x in range(WIDTH):
            row.append(f"'{policy[(x, y)]}'")
        print(" ".join(row))


## Value iteration and policy extraction

Each sweep applies the Bellman update to every non-terminal state:

$$
U_{t+1}(s) = \max_a \Big[ C + \gamma \sum_{s'} P(s' \mid s, a)\, U_t(s') \Big]
$$

The sweeps stop once the largest change in any state is at most `1e-4`. The policy then picks the action with the highest expected utility in each state.

In [3]:
def value_iteration(grid, p, threshold=THRESHOLD, print_each_iteration=False):
    utilities = {s: 0.0 for s in grid.all_states()}
    for t_state, reward in TERMINAL_REWARDS.items():
        utilities[t_state] = reward

    iteration = 0
    while True:
        iteration += 1
        new_utilities = utilities.copy()
        max_diff = 0.0

        for state in grid.all_states():
            if not grid.is_non_sink_state(state):
                continue

            best = max(
                expected_utility(grid, state, action, utilities, p)
                for action in ACTIONS
            )

            max_diff = max(max_diff, abs(best - utilities[state]))
            new_utilities[state] = best

        for t_state, reward in TERMINAL_REWARDS.items():
            new_utilities[t_state] = reward

        utilities = new_utilities

        if print_each_iteration:
            print(f"Iteration {iteration} (max diff={max_diff:.6f}):")
            print_utility_grid(utilities)
            print()

        if max_diff <= threshold:
            break

    return utilities, iteration


def extract_policy(grid, utilities, p):
    policy = {}
    for state in grid.all_states():
        if not grid.is_non_sink_state(state):
            policy[state] = "none"
            continue

        best_action = max(
            ACTIONS,
            key=lambda action: expected_utility(grid, state, action, utilities, p)
        )
        policy[state] = best_action

    return policy


## Solving for p = 0.7

In [4]:
grid = GridWorld()

print("p = 0.7: utilities after each sweep until convergence\n")
utilities_07, iterations_07 = value_iteration(grid, p=0.7, print_each_iteration=True)

print(f"Converged in {iterations_07} iterations.")
print("Final policy (p = 0.7):")
policy_07 = extract_policy(grid, utilities_07, p=0.7)
print_policy_grid(policy_07)


p = 0.7: utilities after each sweep until convergence

Iteration 1 (max diff=0.625000):
  WALL     -0.0400  -1.0000   1.0000
 -0.0400  -0.0400  -0.0400   0.6250
 -0.0400   WALL     -0.0400   WALL   
 -0.0400  -0.0400  -0.0400  -0.0400

Iteration 2 (max diff=0.267425):
  WALL     -0.0780  -1.0000   1.0000
 -0.0780  -0.0780   0.2274   0.7084
 -0.0780   WALL     -0.0780   WALL   
 -0.0780  -0.0780  -0.0780  -0.0780

Iteration 3 (max diff=0.167008):
  WALL     -0.1141  -1.0000   1.0000
 -0.1141   0.0890   0.2774   0.7583
 -0.1141   WALL      0.0890   WALL   
 -0.1141  -0.1141  -0.1141  -0.1141

Iteration 4 (max diff=0.100772):
  WALL     -0.1195  -1.0000   1.0000
 -0.0133   0.1409   0.3345   0.7726
 -0.1484   WALL      0.1699   WALL   
 -0.1484  -0.1484  -0.0133  -0.1484

Iteration 5 (max diff=0.060977):
  WALL     -0.1058  -1.0000   1.0000
  0.0307   0.1855   0.3555   0.7828
 -0.0912   WALL      0.2308   WALL   
 -0.1810  -0.0912   0.0307  -0.0912

Iteration 6 (max diff=0.056863):
  WALL 

## Sweeping p from 0.1 to 0.9

In [5]:
print("Optimal policy for p = 0.1 to 0.9\n")

for i in range(1, 10):
    p = i / 10.0
    utilities, _ = value_iteration(grid, p=p, print_each_iteration=False)
    policy = extract_policy(grid, utilities, p=p)
    print(f"p = {p:.1f}")
    print_policy_grid(policy)
    print()


Optimal policy for p = 0.1 to 0.9

p = 0.1
'none' 'left' 'none' 'none'
'up' 'down' 'down' 'right'
'left' 'none' 'left' 'none'
'right' 'up' 'right' 'up'

p = 0.2
'none' 'left' 'none' 'none'
'up' 'down' 'down' 'right'
'left' 'none' 'up' 'none'
'right' 'up' 'right' 'up'

p = 0.3
'none' 'left' 'none' 'none'
'up' 'down' 'down' 'right'
'up' 'none' 'up' 'none'
'right' 'right' 'right' 'up'

p = 0.4
'none' 'left' 'none' 'none'
'up' 'right' 'down' 'right'
'up' 'none' 'up' 'none'
'up' 'right' 'up' 'left'

p = 0.5
'none' 'left' 'none' 'none'
'right' 'right' 'down' 'right'
'up' 'none' 'up' 'none'
'right' 'right' 'up' 'left'

p = 0.6
'none' 'left' 'none' 'none'
'right' 'right' 'right' 'up'
'up' 'none' 'up' 'none'
'right' 'right' 'up' 'left'

p = 0.7
'none' 'down' 'none' 'none'
'right' 'right' 'right' 'up'
'up' 'none' 'up' 'none'
'right' 'right' 'up' 'left'

p = 0.8
'none' 'down' 'none' 'none'
'right' 'right' 'right' 'up'
'up' 'none' 'up' 'none'
'right' 'right' 'up' 'left'

p = 0.9
'none' 'down' 'non

## What changes as p varies

When `p` is high, the agent goes straight for the goal. From the bottom-left, the policy goes right to `(2, 0)`, up the `x = 2` column to `(2, 2)`, right to `(3, 2)` and then up into the `+1`. At `(1, 3)`, which sits between the wall and the `-1`, the agent steps `down` away from the penalty.

When `p` is low, the move it picks usually doesn't happen. With `p = 0.1`, each perpendicular move is 0.45 likely, while the chosen move is only 0.1 likely. So the best action is often one the agent doesn't expect to make, chosen for where its slips land:

- At `(1, 3)` the policy is `left`, into the wall. The agent almost always slips up or down instead. Up is off the grid, so it stays put, and down takes it to safety. Neither slip can reach the `-1` to its right. A direct `down` would slip sideways and could land on the `-1`.
- At `(2, 2)`, right under the `-1`, the policy is `down` until `p = 0.5`. Its slips go left or right, and right lands next to the `+1`. Choosing `right` or `up` would give the agent a real chance of slipping into the `-1`.
- Once the chosen move is likely enough to happen, these states switch to the direct moves: `(2, 2)` changes to `right` at `p = 0.6`, and `(1, 3)` changes to `down` at `p = 0.7`.

Between `p = 0.4` and `0.6`, the policy changes a few states at a time. In those states the competing actions have close values, so a small change in `p` flips which one wins.

Every move costs `-0.04`, so states far from the goal end up with lower values. That effect is strongest when `p` is low, because reaching the goal takes more moves.